# 14.9 · 多变量 & 多步预测 / Multivariate & Multi-Step Forecasting

> **课程定位 / Where this fits**
> 第 9 课，**Part 14 · 时间序列**。从单序列到多个相互影响的序列, 从预测一步到预测多步。
> Lesson 9, **Part 14 · Time Series**. From one series to multiple interacting ones, from one step to many.
>
> 前面都是**单变量**(只预测一条序列)。但现实常是**多个序列相互影响**——GDP、消费、投资彼此牵动; 一个仓库里成千上万 SKU 的销量相关。**VAR(向量自回归)** 把多个序列**联合建模**:每个序列既用自己的过去, 也用**其它序列的过去**。另外, 实务总要预测**未来多步**(不止下一个), 这有两种策略——**递归**和**直接**, 各有取舍。本课用宏观经济数据搭 VAR 做多变量预测, 并讲清多步预测的两条路。
> So far **univariate** (forecasting one series). But reality often has **multiple interacting series** — GDP, consumption, investment move together; thousands of SKUs in a warehouse correlate. **VAR (Vector Autoregression)** models them **jointly**: each series uses its own past *and* **the others' pasts**. Also, practice always needs **multi-step** forecasts (not just the next), with two strategies — **recursive** and **direct**, each with trade-offs. We build a VAR on macroeconomic data and clarify both multi-step routes.
>
> 💼 **实战/面试视角**："VAR 是什么/和ARIMA区别 / 递归vs直接多步预测的优劣 / 多变量预测的难点" 是多序列预测常考。
> 💼 **Practical/interview angle:** "what VAR is / vs ARIMA / recursive vs direct multi-step / challenges of multivariate" — common.

> 📐 **符号约定 / Notation**
> - VAR(p):每个变量用所有变量过去 p 期预测 / each variable from all variables' past p lags
> - 递归/直接 —— 两种多步预测策略 / recursive / direct multi-step strategies

> 💡 **面试相关 / Interview-relevant**
> - "VAR 和 ARIMA/单变量的区别"（出镜率 ★★★★）
> - "递归多步 vs 直接多步的优劣"（出镜率 ★★★★★）
> - "为什么多步预测比单步难"（★★★★，误差累积）
> - "多变量预测的挑战"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解 VAR 如何联合建模多个相互影响的序列。
   Understand how VAR jointly models interacting series.
2. 在宏观数据上拟合 VAR 并预测。
   Fit a VAR on macro data and forecast.
3. 掌握**递归 vs 直接**两种多步预测策略及取舍。
   Master recursive vs direct multi-step strategies and trade-offs.
4. 理解多步预测为何更难(误差累积)。
   Understand why multi-step is harder (error accumulation).

## 目录 / TOC
1. [多变量:VAR 向量自回归 ⭐](#1)
2. [拟合 VAR 并预测 ⭐](#2)
3. [多步预测:递归 vs 直接 ⭐](#3)
4. [对比 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 多变量:VAR 向量自回归 ⭐ / Multivariate: VAR

很多序列是**相互依赖**的: 消费上升带动 GDP, GDP 又影响投资……单独预测每条序列会**丢掉它们之间的关系**。**VAR(Vector Autoregression, 向量自回归)** 把它们**一起建模**。
Many series are **interdependent**: consumption lifts GDP, which affects investment… forecasting each alone **ignores their relationships**. **VAR (Vector Autoregression)** models them **together**.

它是 AR(自回归, 14.4)的多变量推广: **每个变量的当前值 = 所有变量过去 $p$ 期值的线性组合**。比如 VAR(1) 对两个变量 $x, y$:
It generalizes AR (14.4) to multiple variables: **each variable's current value = a linear combination of all variables' past $p$ lags**. E.g. VAR(1) for two variables $x, y$:

$$x_t = a_1 x_{t-1} + a_2 y_{t-1} + e_t, \qquad y_t = b_1 x_{t-1} + b_2 y_{t-1} + u_t$$

关键: $x_t$ 不只用 $x$ 自己的过去, 还用 $y$ 的过去(反之亦然)——这样就**捕捉了变量间的相互影响**。
Key: $x_t$ uses not only $x$'s own past but also $y$'s past (and vice versa) — **capturing cross-variable influence**.

> 要求(同 ARIMA): VAR 也要求序列**平稳**——非平稳要先差分。我们用 statsmodels 自带的**宏观经济数据**(GDP、消费、投资)。
> Requirement (like ARIMA): VAR also needs **stationary** series — difference first if not. We use statsmodels' bundled **macroeconomic data** (GDP, consumption, investment).


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
import statsmodels.api as sm
sns.set_theme(style="whitegrid")

# statsmodels 自带宏观经济数据(季度) / bundled quarterly macro data
data = sm.datasets.macrodata.load_pandas().data
df = data[["realgdp", "realcons", "realinv"]].copy()      # 实际GDP/消费/投资 / real GDP/consumption/investment
df.index = pd.period_range("1959Q1", periods=len(df), freq="Q").to_timestamp()
print(f"宏观数据: {len(df)} 个季度, 变量 = {list(df.columns)} (GDP/消费/投资, 显然相互影响)")

# 取对数差分使平稳(增长率) / log-difference to stationarity (growth rates)
df_growth = np.log(df).diff().dropna() * 100              # 近似季度增长率(%) / quarterly growth %
fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for ax, col in zip(axes, df.columns):
    ax.plot(df_growth[col]); ax.set_title(f"{col} 季度增长率(%) — 已平稳"); ax.axhline(0, color="gray", ls="--")
plt.tight_layout(); plt.show()
print("三个序列相互牵动(GDP↔消费↔投资); 取对数差分→增长率(平稳), 才能喂给VAR")
print("VAR: 每个变量用'所有变量的过去'预测自己 → 捕捉变量间相互影响(对比单变量各自预测)")


<a id="2"></a>
## 2. 拟合 VAR 并预测 ⭐ / Fit VAR & Forecast

用 statsmodels 拟合 VAR: 先**按 AIC 自动选滞后阶 $p$**(用几期历史), 再预测未来若干季度。VAR 同时输出**所有变量**的预测。
Fit a VAR with statsmodels: first **auto-select the lag order $p$ by AIC** (how many past quarters), then forecast several quarters ahead. VAR outputs forecasts for **all variables** at once.


In [ ]:
from statsmodels.tsa.api import VAR
train, test = df_growth[:-8], df_growth[-8:]              # 按时间切: 留最后8季度测试 / chronological split

model = VAR(train)
lag_sel = model.select_order(maxlags=8)                   # 按信息准则选阶 / select lag order
p = lag_sel.aic                                           # AIC 选出的滞后阶 / AIC-chosen lag
print(f"VAR 按 AIC 选出的滞后阶 p = {p} (即用过去{p}个季度)")
var_fit = model.fit(p)
forecast = var_fit.forecast(train.values[-p:], steps=len(test))   # 多步预测(递归内部完成) / multi-step forecast
fc_df = pd.DataFrame(forecast, index=test.index, columns=df.columns)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col in zip(axes, df.columns):
    train[col][-20:].plot(ax=ax, label="训练"); test[col].plot(ax=ax, label="真实", color="green")
    fc_df[col].plot(ax=ax, label="VAR预测", color="red", ls="--")
    ax.set_title(f"{col} 增长率预测"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"VAR(p={p}) 一次性预测了全部3个变量的未来8季度(利用了变量间的相互依赖)")
print("VAR的forecast内部就是'递归多步': 用预测值滚动着往后推(下面详解多步策略)")


<a id="3"></a>
## 3. 多步预测:递归 vs 直接 ⭐ / Multi-Step: Recursive vs Direct

实务几乎总要预测**未来多步**(预测未来 12 个月, 不止下一个月)。有两种策略(面试高频对比)：
Practice almost always needs **multi-step** forecasts (next 12 months, not just one). Two strategies (high-frequency comparison):
- **递归(recursive / iterated)**:训练一个**单步**模型, 预测下一步 → 把预测**当作真实值**喂回去 → 预测再下一步……反复滚动。(14.6 的 LSTM、VAR 默认都是这样。)
  **Recursive (iterated):** train a **one-step** model; predict the next step → **feed the prediction back as if real** → predict the following step… rolling forward. (LSTM in 14.6 and VAR do this.)
  - 优点: 只需训一个模型, 简单; 缺点: **误差累积**——第一步的错会污染后续输入, 越远越偏。
    Pro: one model, simple; Con: **error accumulation** — early errors corrupt later inputs, drifting over the horizon.
- **直接(direct)**:为**每个预测步长各训一个模型**(一个专门预测 t+1, 一个专门预测 t+2, …, 一个专门预测 t+h), 各自直接用已知历史预测那个特定步长。
  **Direct:** train a **separate model for each horizon** (one for t+1, one for t+2, …, one for t+h), each directly predicting that horizon from known history.
  - 优点: **无误差累积**(每个都用真实历史); 缺点: 要训 $h$ 个模型(贵), 各步长之间预测可能不连贯。
    Pro: **no error accumulation** (each uses real history); Con: $h$ models (costly), forecasts across horizons may be inconsistent.

下面用一个简单回归模型对比两种策略在多步上的表现。
Below we compare both strategies on multi-step with a simple regression model.


In [ ]:
from sklearn.linear_model import LinearRegression
# 用单变量 realgdp 增长率演示递归 vs 直接(便于对比) / demo recursive vs direct on one series
y = df_growth["realgdp"].values
L, H = 8, 8                                               # 用过去8期, 预测未来8步 / lookback 8, horizon 8
split = len(y) - H
y_train, y_test = y[:split], y[split:]

# 递归: 训一个单步模型, 滚动预测 / recursive: one one-step model, roll forward
Xr = np.array([y_train[i:i+L] for i in range(len(y_train)-L)]); Yr = y_train[L:]
rec_model = LinearRegression().fit(Xr, Yr)
window = list(y_train[-L:]); rec_pred = []
for _ in range(H):
    p = rec_model.predict([window[-L:]])[0]; rec_pred.append(p); window.append(p)   # 预测当输入 / feed back

# 直接: 为每个步长h训一个模型(用同样的历史窗口预测t+h) / direct: a model per horizon
dir_pred = []
for h in range(1, H+1):
    Xd = np.array([y_train[i:i+L] for i in range(len(y_train)-L-h+1)])
    Yd = y_train[L+h-1:]                                  # 标签是 t+h / label is t+h
    dm = LinearRegression().fit(Xd, Yd)
    dir_pred.append(dm.predict([y_train[-L:]])[0])        # 都用最后已知窗口预测 / predict from last known window

rec_mae = np.mean(np.abs(np.array(rec_pred)-y_test)); dir_mae = np.mean(np.abs(np.array(dir_pred)-y_test))
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(range(H), y_test, "go-", label="真实")
ax.plot(range(H), rec_pred, "r--s", label=f"递归 (MAE={rec_mae:.2f})")
ax.plot(range(H), dir_pred, "b--^", label=f"直接 (MAE={dir_mae:.2f})")
ax.set_xlabel("预测步长 (越往后越难)"); ax.set_ylabel("GDP增长率"); ax.legend()
ax.set_title("多步预测: 递归(误差累积) vs 直接(各步长独立模型)")
plt.tight_layout(); plt.show()
print(f"递归多步 MAE = {rec_mae:.2f}; 直接多步 MAE = {dir_mae:.2f}")
print("递归: 一个模型简单但误差累积(越远越偏); 直接: 各步长独立模型无累积但要训h个+可能不连贯")
print("实务: 短horizon常用递归(简单); 长horizon或重视远期精度时考虑直接/多输出模型")


<a id="4"></a>
## 4. 对比 + 小结 ⭐ / Summary

```
多变量: 多个序列相互影响(GDP/消费/投资); 单独预测会丢关系 → VAR联合建模
VAR(p): AR的多变量版, 每个变量=所有变量过去p期的线性组合; 捕捉相互依赖; 也要求平稳(先差分)
VAR用途: 多序列联合预测 + 脉冲响应分析 + Granger因果(14.11); 一次输出所有变量预测
多步预测两策略:
  递归(recursive): 单步模型滚动预测(预测当输入); 简单但误差累积(越远越偏)
  直接(direct): 每个步长训独立模型; 无累积但要训h个+可能不连贯
为什么多步难: 递归的误差累积 + 远期本身不确定性大
实务: 短horizon用递归; 长horizon/重远期精度考虑直接, 或用多输出模型(seq2seq一次出多步)
```

### 💡 面试速查 / Interview cheat-sheet
1. **VAR**: AR多变量版, 每变量用所有变量的过去; 捕捉相互依赖; 需平稳。
   VAR: multivariate AR, each variable from all variables' pasts; captures interdependence; needs stationarity.
2. **递归多步**: 一个单步模型滚动(预测当输入); 简单但误差累积。
   Recursive: one one-step model rolled forward; simple but errors compound.
3. **直接多步**: 每步长一个模型; 无累积但训h个+可能不连贯。
   Direct: a model per horizon; no compounding but h models + possible inconsistency.
4. **多步为何难**: 误差累积 + 远期不确定性大。
   Why hard: error accumulation + far-horizon uncertainty.
5. **选择**: 短horizon递归; 长horizon直接/多输出(seq2seq)。
   Choice: short → recursive; long → direct/multi-output.

### 下一节 / Next
**14.10 时序异常检测**——在时间序列里找"不正常"的点(服务器故障、欺诈交易、设备异常)。核心思路: 先用分解/预测得到"正常应该是什么样", 再看**残差**(实际与预期的偏差), 偏离太大就是异常。我们会用 STL 残差 + 统计方法做检测。
**14.10 TS Anomaly Detection** — find "abnormal" points in a series (server failures, fraud, equipment faults). Core idea: model "what's normal" via decomposition/forecasting, then watch the **residual** (actual vs expected); large deviations are anomalies. We'll use STL residuals + statistical methods.
